Copyright (c) Microsoft Corporation. All rights reserved. 

Licensed under the MIT License.

# Use FLAML to Tune ChatGPT

FLAML offers a cost-effective hyperparameter optimization technique [EcoOptiGen](https://arxiv.org/abs/2303.04673) for tuning Large Language Models. Our study finds that tuning hyperparameters can significantly improve the utility of LLMs.

In this notebook, we tune OpenAI ChatGPT (both GPT-3.5 and GPT-4) models for math problem solving. We use [the MATH benchmark](https://crfm.stanford.edu/helm/latest/?group=math_chain_of_thought) for measuring mathematical problem solving on competition math problems with chain-of-thoughts style reasoning. 

## Requirements

FLAML requires `Python>=3.7`. To run this notebook example, please install flaml with the `[synapse,openai]` option:
```bash
pip install flaml[synapse,openai]
```

In [ ]:
%pip install "flaml[synapse,openai]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl" datasets

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 8, Finished, Available)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.1/275.1 kB 2.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.6/474.6 kB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 101.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 36.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 24.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 94.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 41.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 86.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 87.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 32.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 28.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 54.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

In [ ]:
# Setup OpenAI
import openai
from synapse.ml.mlflow import get_mlflow_env_config

mlflow_env_configs = get_mlflow_env_config()
access_token = mlflow_env_configs.driver_aad_token
baseurl = mlflow_env_configs.workload_endpoint + "cognitive/openai/"

openai.api_key = mlflow_env_configs.driver_aad_token
openai.api_base = baseurl
openai.api_type = "azure_ad"
openai.api_version = "2023-03-15-preview"

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 21, Finished, Available)

### Import the oai and tune subpackages from flaml

FLAML has provided an API for hyperparameter optimization of OpenAI ChatGPT models: `oai.ChatCompletion.tune` and to make a request with the tuned config: `oai.ChatCompletion.create`. First, we import oai from flaml:

In [ ]:
from flaml import oai, tune

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 22, Finished, Available)

### Set your API Endpoint

The [`config_list_openai_aoai`](https://microsoft.github.io/FLAML/docs/reference/autogen/oai/openai_utils#config_list_openai_aoai) function tries to create a list of  Azure OpenAI endpoints and OpenAI endpoints. It assumes the api keys and api bases are stored in the corresponding environment variables or local txt files:

- OpenAI API key: os.environ["OPENAI_API_KEY"] or `openai_api_key_file="key_openai.txt"`.
- Azure OpenAI API key: os.environ["AZURE_OPENAI_API_KEY"] or `aoai_api_key_file="key_aoai.txt"`. Multiple keys can be stored, one per line.
- Azure OpenAI API base: os.environ["AZURE_OPENAI_API_BASE"] or `aoai_api_base_file="base_aoai.txt"`. Multiple bases can be stored, one per line.

It's OK to have only the OpenAI API key, or only the Azure Open API key + base.


In [ ]:
# config_list = oai.config_list_openai_aoai()

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 23, Finished, Available)

The config list looks like the following:
```python
config_list = [
    {'api_key': '<your OpenAI API key here>'},  # only if OpenAI API key is found
    {
        'api_key': '<your first Azure OpenAI API key here>',
        'api_base': '<your first Azure OpenAI API base here>',
        'api_type': 'azure',
        'api_version': '2023-03-15-preview',
    },  # only if the at least one Azure OpenAI API key is found
    {
        'api_key': '<your second Azure OpenAI API key here>',
        'api_base': '<your second Azure OpenAI API base here>',
        'api_type': 'azure',
        'api_version': '2023-03-15-preview',
    },  # only if the second Azure OpenAI API key is found
]
```

You can directly override it if the above function returns an empty list, i.e., it doesn't find the keys in the specified locations.

## Load dataset

We load the competition_math dataset. The dataset contains 201 "Level 2" Algebra examples. We use a random sample of 20 examples for tuning the generation hyperparameters and the remaining for evaluation.

In [ ]:
import datasets

seed = 41
data = datasets.load_dataset("competition_math")
train_data = data["train"].shuffle(seed=seed)
test_data = data["test"].shuffle(seed=seed)
n_tune_data = 20
tune_data = [
    {
        "problem": train_data[x]["problem"],
        "solution": train_data[x]["solution"],
    }
    for x in range(len(train_data)) if train_data[x]["level"] == "Level 2" and train_data[x]["type"] == "Algebra"
][:n_tune_data]
test_data = [
    {
        "problem": test_data[x]["problem"],
        "solution": test_data[x]["solution"],
    }
    for x in range(len(test_data)) if test_data[x]["level"] == "Level 2" and test_data[x]["type"] == "Algebra"
]
print(len(tune_data), len(test_data))


StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 24, Finished, Available)

Found cached dataset competition_math (/home/trusted-service-user/.cache/huggingface/datasets/competition_math/default/1.0.0/2a2a2995c2847186883ecd64f69be7d602b8a6f6b51950624d4dc2263f93333b)


  0%|          | 0/2 [00:00<?, ?it/s]

Loading cached shuffled indices for dataset at /home/trusted-service-user/.cache/huggingface/datasets/competition_math/default/1.0.0/2a2a2995c2847186883ecd64f69be7d602b8a6f6b51950624d4dc2263f93333b/cache-f1cfe8228271b121.arrow
Loading cached shuffled indices for dataset at /home/trusted-service-user/.cache/huggingface/datasets/competition_math/default/1.0.0/2a2a2995c2847186883ecd64f69be7d602b8a6f6b51950624d4dc2263f93333b/cache-d155a2d38c23bd53.arrow
20 201


Check a tuning example:

In [ ]:
print(tune_data[1]["problem"])

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 25, Finished, Available)

A student accidentally added five to both the numerator and denominator of a fraction, changing the fraction's value to $\frac12$. If the original numerator was a 2, what was the original denominator?


Here is one example of the canonical solution:

In [ ]:
print(tune_data[1]["solution"])

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 26, Finished, Available)

Let $d$ be the original denominator.  After adding 5 to both numerator and denominator, the fraction becomes $\frac{7}{d+5}$.  If a fraction with 7 in the numerator is equivalent to $\frac{1}{2}$, then the denominator is 14.  Solving $d+5=14$, we find $d=\boxed{9}$.


## Define Success Metric

Before we start tuning, we need to define the success metric we want to optimize. For each math task, we use voting to select a response with the most common answers out of all the generated responses. If it has an equivalent answer to the canonical solution, we consider the task as successfully solved. Then we can optimize the mean success rate of a collection of tasks.

In [ ]:
from flaml.autogen.math_utils import eval_math_responses

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 27, Finished, Available)

## Use the tuning data to find a good configuration


For (local) reproducibility and cost efficiency, we cache responses from OpenAI.

In [ ]:
oai.ChatCompletion.set_cache(seed)

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 28, Finished, Available)

This will create a disk cache in ".cache/{seed}". You can change `cache_path` in `set_cache()`. The cache for different seeds are stored separately.

### Perform tuning

The tuning will take a while to finish, depending on the optimization budget. The tuning will be performed under the specified optimization budgets.

* `inference_budget` is the target average inference budget per instance in the benchmark. For example, 0.004 means the target inference budget is 0.004 dollars, which translates to 2000 tokens (input + output combined) if the gpt-3.5-turbo model is used.
* `optimization_budget` is the total budget allowed to perform the tuning. For example, 1 means 1 dollars are allowed in total, which translates to 500K tokens for the gpt-3.5-turbo model.
* `num_sumples` is the number of different hyperparameter configurations which is allowed to try. The tuning will stop after either num_samples trials or after optimization_budget dollars spent, whichever happens first. -1 means no hard restriction in the number of trials and the actual number is decided by `optimization_budget`.

Users can specify tuning data, optimization metric, optimization mode, evaluation function, search spaces etc.. The default search space is:

```python
default_search_space = {
    "model": tune.choice([
        "gpt-3.5-turbo",
        "gpt-4",
    ]),
    "temperature_or_top_p": tune.choice(
        [
            {"temperature": tune.uniform(0, 2)},
            {"top_p": tune.uniform(0, 1)},
        ]
    ),
    "max_tokens": tune.lograndint(50, 1000),
    "n": tune.randint(1, 100),
    "prompt": "{prompt}",
}
```

The default search space can be overridden by users' input.
For example, the following code specifies a fixed prompt template. For hyperparameters which don't appear in users' input, the default search space will be used.

In [ ]:
import logging

prompts = ["{problem} Solve the problem carefully. Simplify your answer as much as possible. Put the final answer in \\boxed{{}}."]
config, analysis = oai.ChatCompletion.tune(
    data=tune_data,  # the data for tuning
    metric="success_vote",  # the metric to optimize
    mode="max",  # the optimization mode
    eval_func=eval_math_responses,  # the evaluation function to return the success metrics
    # log_file_name="logs/math.log",  # the log file name
    inference_budget=0.02,  # the inference budget (dollar per instance)
    optimization_budget=1,  # the optimization budget (dollar in total)
    # num_samples can further limit the number of trials for different hyperparameter configurations;
    # -1 means decided by the optimization budget only
    num_samples=20,
    # model="gpt-3.5-turbo",  # comment to tune both gpt-3.5-turbo and gpt-4
    prompt=prompts,  # the prompt templates to choose from
    # stop="###",  # the stop sequence
    # config_list=config_list,  # the endpoint list, not needed in Fabric
    # logging_level=logging.INFO,  # the logging level
)


StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 31, Finished, Available)

[I 2023-05-30 14:36:23,885] A new study created in memory with name: optuna
[I 2023-05-30 14:36:23,889] A new study created in memory with name: optuna


No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
[flaml.tune.tune: 05-30 14:36:25] {807} INFO - trial 1 config: {'prompt': 0, 'subspace': {'model': 'gpt-4', 'max_tokens': 347, 'temperature_or_top_p': {'temperature': 0.7373189005362395}, 'n': 1}}
[flaml.tune.tune: 05-30 14:36:26] {205} INFO - result: {'expected_success': 0.8, 'success': 0.8, 'success_vote': 0.8, 'voted_answer': 'We have a difference of squares, so we can factor as follows:\n\n\\begin{align*}\nt^2-121 &= t^2-11^2 \\\\\n&= (t+11

### Output tuning results

After the tuning, we can print out the config and the result found by FLAML:

In [ ]:
print("optimized config", config)
print("best result on tuning data", analysis.best_result)

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 32, Finished, Available)

optimized config {'prompt': '{problem} Solve the problem carefully. Simplify your answer as much as possible. Put the final answer in \\boxed{{}}.', 'max_tokens': 433, 'n': 54, 'model': 'gpt-3.5-turbo', 'top_p': 0.7145757833976906}
best result on tuning data {'expected_success': 0.9999983615080609, 'success': 1.0, 'success_vote': 0.9, 'voted_answer': 'We notice that $t^2-121$ is a difference of squares: $$t^2-121 = (t+11)(t-11).$$ Therefore, the prime factorization of $t^2-121$ is $$(t+11)(t-11) = \\boxed{(t+11)(t-11)}.$$', 'votes': 41.5, 'total_cost': 0.6937460000000001, 'cost': 0.3052160000000001, 'inference_cost': 0.0144278, 'training_iteration': 0, 'config': {'prompt': 0, 'subspace': {'max_tokens': 433, 'temperature_or_top_p': {'top_p': 0.7145757833976906}, 'n': 54, 'model': 'gpt-3.5-turbo'}}, 'config/prompt': 0, 'config/subspace': {'max_tokens': 433, 'temperature_or_top_p': {'top_p': 0.7145757833976906}, 'n': 54, 'model': 'gpt-3.5-turbo'}, 'experiment_tag': 'exp', 'time_total_s': 

### Make a request with the tuned config

We can apply the tuned config on the request for an example task:

In [ ]:
response = oai.ChatCompletion.create(context=tune_data[1], config_list=config_list, **config)
metric_results = eval_math_responses(oai.ChatCompletion.extract_text(response), **tune_data[1])
print("response on an example data instance:", response)
print("metric_results on the example data instance:", metric_results)


StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 33, Finished, Available)

response on an example data instance: {
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Let the original denominator be $d$. Then the original fraction is $\\frac{2}{d}$. When 5 is added to both the numerator and denominator, the new fraction is $\\frac{2+5}{d+5}=\\frac{7}{d+5}$. We know that this new fraction is equal to $\\frac{1}{2}$, so we can set up the equation $\\frac{7}{d+5}=\\frac{1}{2}$ and solve for $d$.\n\nMultiplying both sides by $2(d+5)$ gives $14= d+5$. Subtracting 5 from both sides gives $d=9$.\n\nTherefore, the original fraction was $\\frac{2}{9}$, and our final answer is $\\boxed{9}$.",
        "role": "assistant"
      }
    },
    {
      "finish_reason": "stop",
      "index": 1,
      "message": {
        "content": "Let the original denominator be $x$. Then, the original fraction was $\\frac{2}{x}$.\n\nWhen the student added 5 to both the numerator and denominator, the new fraction became $\\frac{2+5}{

### Evaluate the success rate on the test data

You can use flaml's `oai.ChatCompletion.test` to evaluate the performance of an entire dataset with the tuned config. The following code will take a while (30 mins to 1 hour) to evaluate all the test data instances if uncommented and run. It will cost roughly $3. 

In [ ]:
# result = oai.ChatCompletion.test(test_data, logging_level=logging.INFO, config_list=config_list, **config)
# print("performance on test data with the tuned config:", result)

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 34, Finished, Available)

What about the default, untuned gpt-4 config (with the same prompt as the tuned config)? We can evaluate it and compare:

In [ ]:
# the following code will cost roughly $2 if uncommented and run.

# default_config = {"model": 'gpt-4', "prompt": prompts[0]}
# default_result = oai.ChatCompletion.test(test_data, config_list=config_list, **default_config)
# print("performance on test data from gpt-4 with a default config:", default_result)

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 35, Finished, Available)

In [ ]:
# print("tuned config succeeds in {:.1f}% test cases".format(result["success_vote"] * 100))
# print("untuned config succeeds in {:.1f}% test cases".format(default_result["success_vote"] * 100))

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 36, Finished, Available)

The default use of GPT-4 has a much lower accuracy. Note that the default config has a lower inference cost. What if we heuristically increase the number of responses n?

In [ ]:
# The following evaluation costs $3 and longer than one hour if you uncomment it and run it.

# config_n2 = {"model": 'gpt-4', "prompt": prompts[0], "n": 2}
# result_n2 = oai.ChatCompletion.test(test_data, config_list=config_list, **config_n2)
# print("performance on test data from gpt-4 with a default config and n=2:", result_n2)


StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 37, Finished, Available)

The inference cost is doubled and matches the tuned config. But the success rate doesn't improve much. What if we further increase the number of responses n to 5?

In [ ]:
# The following evaluation costs $8 and longer than one hour if you uncomment it and run it.

# config_n5 = {"model": 'gpt-4', "prompt": prompts[0], "n": 5}
# result_n5 = oai.ChatCompletion.test(test_data, config_list=config_list, **config_n5)
# print("performance on test data from gpt-4 with a default config and n=5:", result_n5)

StatementMeta(, f3750b55-1be5-4588-96ca-f50f76201e83, 38, Finished, Available)

We find that the 'success_vote' metric is increased at the cost of exceeding the inference budget. But the tuned configuration has both higher 'success_vote' (91% vs. 87%) and lower average inference cost ($0.015 vs. $0.037 per instance).

A developer could use flaml to tune the configuration to satisfy the target inference budget while maximizing the value out of it.